# Anatomical Magnetic Resonance Imaging (MRI) data

This session will guide you through the steps that one needs to follow in order to understand and work with neuroimaging data. We will cover how this data is obtained and what can be seen from it. NeuroImaging data is sensitive, heavy, complex and, most importantly, collaborative. Thus, it needs to be stored in efficient and standardized forms: the NIFTI and BIDS formats. Unfortunately MR scaners do not produce these types of formats and data usually needs to be semiautomatically transformed in order to meet the aforementioned criteria.

You will also learn what softwares and/or custom codes one can use in order to inspect the images once they are in the correct format. In order to know what can be done with this type of data is perhaps more important to be aware at what you are not looking at. Understanding what is the data showing you is a crucial step prior to digging into the specifics of the data structure and what can be done with it.

Eventually, you will be in a position where you can do what you came here to do... use some machine learning or artificial intelligence methods to unravel what the naked eye might have missed. One of the gold-standard of medical imaging is the segmentation of different structures or lesions that can be seen in the brain. Only after doing a manual segmentation can one see the huge advantage of letting stochastic gradient descent do the heavy work.

The notebook is organized as follows
1.   From the MR scanner to your laptop: DICOM, NIFTI and BIDS
2.   Data and visualization: What do I have? How do I see what I have?
3.   Tissue segmentation: Unsupervised vs supervised methods
4.   Anatomical MRI acquisitions: What am I looking at?
5.   Homework

Just to be on the safe do what you probably already know:
```
conda create -y -n ML4Neuro python=3.11 numpy matplotlib
conda init bash
conda activate ML4Neuro
conda install -c conda-forge jupyterlab
cd /path/to/github/directory
jupyter-lab
```

## 1. From the MR scanner to your laptop: DICOM, NIFTI and BIDS.

The *Digital Imaging and Communications in Medicine* (DICOM) is the standard format for storing and transmitting medical imaging data and information. In this section we will take a look at what this format offers as well as its shortcomings. Ultimately, we need to convert from DICOM to NIFTI and reorganize our data in a way that it's compliant with the BIDS format. This organizational format ensures a correct usage of neuroimaging software increasing the reproducibility and replicability of neuroimaging findings.


In [ ]:
import os, glob, json, subprocess
import pydicom as dicom
import matplotlib.pylab as plt
import numpy as np
import dicom2nifti
import nibabel as nib
from nilearn import plotting
os.environ["FSLDIR"] = os.path.expanduser("~/fsl")
os.environ["FSLOUTPUTTYPE"] = "NIFTI_GZ"
os.environ["PATH"] = os.environ["FSLDIR"] + "/share/fsl/bin:" + os.environ["PATH"]


In [ ]:
# Prepare the directory variables
DICOM_subject = 'FCS-02AMC'
DICOM_acq = '4'
DICOM_dir = f'./DICOM-dir/{DICOM_subject}/{DICOM_acq}/' 
slices = sorted(glob.glob(DICOM_dir+'DICOM/*.dcm'))

In [ ]:
# Read a single DICOM image
dicom_image = dicom.dcmread(slices[0])
dicom_image

This is a lot of information, which may or may not be useful depending on the experiment or on the type of paper your are trying to write. Given that ``` dicom_image ``` is a python object, we can access whatever fields we wish. Several acquisition parameters are usually reported in the studies and all of them can easily be accessed through this object. It is common to have the same scanning protocols for all the subjects in the dataset, but sometimes mistakes occur. With this type of software it should be easy to automatize searching schemes in order to stablish whether the protocol was changed during a study.



In [ ]:
# What was the Repetition Time of the protocol?
TR = dicom_image.RepetitionTime
print(f"Repetition Time = {TR} (ms)")
# What was the Echo Time of the protocol?
ET = dicom_image.EchoTime
print(f"Echo Time = {ET} (ms)")
# What was the Type of acquisition?
ACQ_type = dicom_image.SeriesDescription
print(f"Acquisition type: {ACQ_type}")

However, can we acess the data? If so, what is the data exactly?

In [ ]:
data = dicom_image.pixel_array
print(f"The shape of the data is: {data.shape}")
print(f"The type of variable is: {type(data)}")
print(data)

**Question:** What are the main differences between acquisition '16' and '4' of the subject with ID 'FCS-02AMC'?

**Answer:**

| | Acquisition 4 | Acquisition 16 |
|---|---|---|
| Type | `MPR_1mm_iso` - T1-weighted anatomical (MPRAGE, GR/IR) | `dti_63dir_2mm_iso` - diffusion-weighted (EPI), 63 directions |
| Data | 3D volume, 256x256x176, 1 mm isotropic | 4D volume, 128x128x64x64, 2 mm isotropic (stored as 1024x1024 mosaics) |
| TR / TE / flip angle | 1950 ms / 2.26 ms / 9 deg | 9200 ms / 92 ms / 90 deg |
| DICOM files | 176 (one per slice) | 64 (one per gradient direction) |

Acquisition 4 shows anatomy (good GM/WM contrast, high resolution), acquisition 16 measures water diffusion (lower resolution, needs b-values and b-vectors).

It is clear that the DICOM format stores a lot of information including the data itself. Yet, what is the data? How can you see what you have stored? And how compact and easy to use is this format when it comes to numerical analyses of the scans?

In [ ]:
 
num_to_plot = 3

fig, axes = plt.subplots(num_to_plot, num_to_plot)
if num_to_plot > 1:
  axes = axes.flatten()
  for i, ax in enumerate(axes):
    ax.imshow(
        dicom.dcmread(slices[i]).pixel_array, cmap='gray'
    )
    ax.set_xticks([]), ax.set_yticks([])
else:
  axes.imshow(
      dicom.dcmread(slices[35]).pixel_array, cmap='gray'
  )
  axes.set_xticks([]), axes.set_yticks([])
plt.show()

Ares these images the same? Recall that the brain (or any other organ) is a 3D volume, and current MR scanners take 2D pictures hence they have to move along the head volume capturing *slices* of the subject. The DICOM format stores all these slices together with the corresponding information for each slice (e.g., Window Center). Each slice is stored separately in the acquisition folder (e.g., 16) with a very strange and almost undesciferable name. Furthermore, we can only visualize a single viewing plane that depends on the acquisition parameters. In the former example we could plot the sagital plane straight from the data. Can we do better? Yes!

The *Neuroimaging Informatics Technology Initiative* (NIfTI) format is commonly used brain MR data and it allows to store all the desired information available in the DICOM directory into a single file. Importantly, there is a standard file-naming code that should be taken into consideration: **the FILE-NAME should not be left unattended!** For every subject, a whole DICOM session will be condensed into:

*   a FILE-NAME.nii file containing the data
*   a FILE-NAME.json file containing the aquisition parameters
*   a FILE-NAME.bvec file containing gradient directions
*   a FILE-NAME.bval file containing the gradient strengths

Importantly, the two last files will only appear in the case of difussion MRI acquisitions. For now let's ignore their meaning since we will focus on them in the next laboratory session.

To move from DICOM to NIFTI formats one could build a custom converter. The basics would require a stacking of all slices along a 3rd dimension ensuring that each slice is positioned correctly along the depth axis. A proper json file should also be manually written with the information extracted from the DICOM heather. Evenmore, in the case of diffusion data, the gradient strength and orientation would also need to be decoded from the DICOM heather. Yet, as you can probably imagine, there is a better and faster way to do all this. For now, let's convert the previous DICOM session to NIFTI and see what it looks like.

In [ ]:
T1_DICOM = f'./DICOM-dir/{DICOM_subject}/4/'
DTI_DICOM = f'./DICOM-dir/{DICOM_subject}/16/'

NIFTI_dir = f'./subject_{DICOM_subject}_NIFTI_dir/pydicom/'
if not os.path.exists(NIFTI_dir):
  os.makedirs(NIFTI_dir)

# Apply conversion
dicom2nifti.convert_directory(T1_DICOM, NIFTI_dir, compression=True)
dicom2nifti.convert_directory(DTI_DICOM, NIFTI_dir, compression=True)

Where are the ```.json``` files? Have we lied to you before? No!

***Exercise*** *Find a way to obtain the missing ```.json``` with the ```pydicom``` library. The documentation can be found [here](https://icometrix.github.io/dicom2nifti/).


In [ ]:
# Exercise: build the missing .json sidecars from the DICOM header with pydicom.
# dicom2nifti names each output "<acquisition>_<description>.nii.gz", so the
# acquisition number tells us which DICOM series the sidecar has to come from.
for nii in sorted(glob.glob(NIFTI_dir + '*.nii.gz')):
    acq = os.path.basename(nii).split('_')[0]
    ds = dicom.dcmread(sorted(glob.glob(f'./DICOM-dir/{DICOM_subject}/{acq}/DICOM/*.dcm'))[0])
    sidecar = {
        "Modality": ds.Modality,
        "Manufacturer": ds.Manufacturer,
        "ManufacturersModelName": ds.ManufacturerModelName,
        "MagneticFieldStrength": float(ds.MagneticFieldStrength),
        "SeriesDescription": ds.SeriesDescription,
        "ScanningSequence": "_".join(ds.ScanningSequence) if ds["ScanningSequence"].VM > 1 else ds.ScanningSequence,
        "SequenceName": ds.SequenceName,
        "RepetitionTime": float(ds.RepetitionTime) / 1000,  # DICOM: ms, BIDS: s
        "EchoTime": float(ds.EchoTime) / 1000,
        "FlipAngle": float(ds.FlipAngle),
        "SliceThickness": float(ds.SliceThickness),
        "PixelSpacing": [float(p) for p in ds.PixelSpacing],
    }
    json_file = nii.replace('.nii.gz', '.json')
    with open(json_file, 'w') as f:
        json.dump(sidecar, f, indent=4)
    print(f"Saved {json_file}")

print(json.dumps(sidecar, indent=4))

Have you found a way to do so? If you did, please show it to us. If you didn't succeed do not worry too much because we don't know how to do it either.

Besides, you may have noticed that the FILE-NAME is a strange combination of letters and numbers. In fact, if you try to convert other dicom series you will notice how the name is identical. While this indicates that the protocol for every subject was kept the same, it can be problematic when it comes to identifying subjects and working with the whole dataset.The proper way to do so is to work with LINUX/MACOS, Windows is not really an option: 

![windows](./windows_support.PNG)

Once you have proper operating system you can proceed. Alternatively there are virtual machines or docker containers with most of the necessary neuroimaging software which might work properly on Windows: [Neurodesk](https://www.neurodesk.org/) for example. Anyhow, the best way to convert between formats is using [dmc2niix](https://manpages.ubuntu.com/manpages/jammy/en/man1/dcm2niix.1.html).

In [ ]:
!pip install dcm2niix # This will install the software package but will only be available within the current conda environment
# sudo apt install dcm2niix will install it system wide in Linux (requires sudo password)

In [ ]:
%%bash

DICOM_dir="./DICOM-dir/FCS-02AMC/"
NIFTI_dir_dcm2niix="./subject_FCS-02AMC_NIFTI_dir/dcm2niix/"
mkdir -p $NIFTI_dir_dcm2niix

dcm2niix -w 1 -o $NIFTI_dir_dcm2niix $DICOM_dir

As you can see the ```.json``` files are automatically generated and the conversion is much faster: we converted a whole subject in less than 0.5s. It is recommended to zip the .nii files using the ```-z``` flag. Furthermore, you have more control on the output name if you use the ```-f``` flag. You can customize your name in any way you prefer so that you can easily identify the files you converted.

In [ ]:
%%bash

DICOM_dir="./DICOM-dir/FCS-02AMC/"
NIFTI_dir_dcm2niix_naming="./subject_FCS-02AMC_NIFTI_dir/dcm2niix-naming/"
mkdir -p $NIFTI_dir_dcm2niix_naming

dcm2niix -w 1 -o $NIFTI_dir_dcm2niix_naming -f %n--%p--%t $DICOM_dir 

Don't worry you don't have to remember everything we say. Doing research involves a lot of google search queries, so one basically becomes very profficient and precise about the words used in the search bar. Here are a couple of useful links that might come in handy: [BIDS](https://bids.neuroimaging.io/) and [dcm2niix](https://manpages.ubuntu.com/manpages/jammy/en/man1/dcm2niix.1.html).

***Exercise***: *Try to convert the whole DICOM directory to NIFTI files and reorganize them in a way that is BIDS compliant. You can reorganize them manually or automatically using regular expressions and the ```os``` library. To be sure the final dataset agrees with the minimum BIDS requirements you can use any of the two following options:*

1. The python interface (although it is not the best): ```pip install bids-validator```
2. The web interface [bids-validator](http://bids-standard.github.io/bids-validator/)

Be sure to include all the necessary information when converting your dicom files so that you can identify them properly. 

![bids](./BIDS.jpg)

In [ ]:
# TODO: Exercise


DICOM_root = "./DICOM-dir"
BIDS_root = "./BIDS-dir"

os.makedirs(BIDS_root, exist_ok=True)

with open(f"{BIDS_root}/dataset_description.json", "w") as f:
    json.dump({"Name": "CORBETTA head study", "BIDSVersion": "1.8.0"}, f, indent=4)


for i, subject_raw in enumerate(sorted(os.listdir(DICOM_root)), start=1):
    sub_label = f"{i:02d}"
    for series_dir in sorted(glob.glob(f"{DICOM_root}/{subject_raw}/*")):
        dcm_files = sorted(glob.glob(f"{series_dir}/DICOM/*.dcm"))
        if not dcm_files:
            continue
        desc = dicom.dcmread(dcm_files[0]).SeriesDescription.lower()

        if "mpr" in desc:
            modality, suffix = "anat", "T1w"
        elif "dti" in desc:
            modality, suffix = "dwi", "dwi"
        else:
            continue

        out_dir = f"{BIDS_root}/sub-{sub_label}/{modality}"
        os.makedirs(out_dir, exist_ok=True)
        subprocess.run([
            "dcm2niix", "-b", "y", "-z", "y", "-w", "1",
            "-f", f"sub-{sub_label}_{suffix}",
            "-o", out_dir, f"{series_dir}/DICOM",
        ])


In [ ]:
# 1) the python package or the web-interface or,
from bids_validator import BIDSValidator

# is_bids() checks one path (relative to the dataset root), so check every file
BIDS_root = os.path.abspath("./BIDS-dir")
validator = BIDSValidator()
for path in sorted(glob.glob(f"{BIDS_root}/sub-*/*/*")):
    rel_path = "/" + os.path.relpath(path, BIDS_root)
    print(validator.is_bids(rel_path), rel_path)

Sometimes, not all the data is available or useful to us. However, some files are required in order to have a more transparent and ready-to-use dataset. Remember, people are busy so you need to make things friendly for the user. Transparency and replicability are also encouraged even jeopardizing your own publication rates. For your own personal reasons or because the experiment demands it, you can also add a .bidsignore file pointing to files or directories that could be useful. The rationale is the same as the typical .gitignore file. For example, in lesion studies, it might be useful to have a directory with all the lesion masks even if its existence is not contemplated in the BIDS formatting.

**Question:** What files are you missing? What information should these files have?

***Optional exercise:*** *Add the necessary files (invent data if necessary) so that your directory complies with the validator's demands.*

**Answer:** The validator initially reported a missing `README` and no `Authors` in `dataset_description.json`. I added both, plus `participants.tsv` / `participants.json`, which map the BIDS labels (`sub-01`...) to the original DICOM IDs (`FCS-02AMC`...) - otherwise this information would be lost after renaming.

- `README` - what the dataset is, how it was converted, known issues.
- `dataset_description.json` - name, BIDS version, authors (required).
- `participants.tsv` - one row per subject (ID, and demographics such as age/sex if available).

The remaining warnings come from the data itself, not from the conversion: `sub-02` (`fcs-01acm`) has no diffusion series in the DICOM directory, and the diffusion parameters differ between subjects. I did not invent a DWI scan for sub-02; the gap is documented in the `README`.

### Output from validator:

```
❯ npx bids-validator ./BIDS-dir

bids-validator@1.15.0
    1: [WARN] Not all subjects contain the same files. Each subject should contain the same number of files with the same naming unless some files are known to be missing. (code: 38 - INCONSISTENT_SUBJECTS)
        ./sub-02/dwi/sub-02_dwi.bval
            Evidence: Subject: sub-02; Missing file: sub-02_dwi.bval
        ./sub-02/dwi/sub-02_dwi.bvec
            Evidence: Subject: sub-02; Missing file: sub-02_dwi.bvec
        ./sub-02/dwi/sub-02_dwi.json
            Evidence: Subject: sub-02; Missing file: sub-02_dwi.json
        ./sub-02/dwi/sub-02_dwi.nii.gz
            Evidence: Subject: sub-02; Missing file: sub-02_dwi.nii.gz

    Please visit https://neurostars.org/search?q=INCONSISTENT_SUBJECTS for existing conversations about this issue.

    2: [WARN] Not all subjects/sessions/runs have the same scanning parameters. (code: 39 - INCONSISTENT_PARAMETERS)
        ./sub-01/dwi/sub-01_dwi.nii.gz
        ./sub-04/dwi/sub-04_dwi.nii.gz

    Please visit https://neurostars.org/search?q=INCONSISTENT_PARAMETERS for existing conversations about this issue.

    3: [WARN] The Authors field of dataset_description.json should contain an array of fields - with one author per field. This was triggered based on the presence of only one author field. Please ignore if all contributors are already properly listed. (code: 102 - TOO_FEW_AUTHORS)

    Please visit https://neurostars.org/search?q=TOO_FEW_AUTHORS for existing conversations about this issue.

        Summary:                  Available Tasks:        Available Modalities:
        30 Files, 315.14MB                                MRI
        5 - Subjects
        1 - Session


    If you have any questions, please post on https://neurostars.org/tags/bids.
```

In [ ]:
p = "BIDS-dir/dataset_description.json"
d = json.load(open(p))
d["Authors"] = ["Dominik Marek"]
json.dump(d, open(p, "w"), indent=4)

# participants.tsv: keeps the link between BIDS labels and the original DICOM IDs
# (same ordering as the conversion loop above)
with open("BIDS-dir/participants.tsv", "w") as f:
    f.write("participant_id\tsource_id\n")
    for i, subject_raw in enumerate(sorted(os.listdir("./DICOM-dir")), start=1):
        f.write(f"sub-{i:02d}\t{subject_raw}\n")

with open("BIDS-dir/participants.json", "w") as f:
    json.dump({"source_id": {"Description": "Subject ID in the original DICOM directory"}}, f, indent=4)

print(open("BIDS-dir/participants.tsv").read())

To understand the advantages that the NIFTI and BIDS formats offer it is enough to install and import a single python library. [NiBabel](https://nipy.org/nibabel/) The BIDS-dir folder has already some subjects with the correct organization. Feel free to take a look or use the one you created in the last step.

In [ ]:
nifti = "./BIDS-dir/sub-01/anat/sub-01_T1w.nii.gz"
img1 = nib.load(nifti)
header1, affine1, data1 = img1.header, img1.affine, img1.get_fdata()

Nibabel loads all the information present in the NIFTI file onto a python object with three attributes:

1. The **nifti header** contains all the information necessary to interpret the data. It can be used to quickly inspect the number of dimensions, the voxel size, the orientation of the image, ... As opposed to the PyDicom object, the header in NiBabel is a dictionary.

In [ ]:
print(header1)

2. The **nifti affine** is a 4x4 matrix that encodes the spatial interpretation of the data. Each voxel $(i,j,k)$ in the data represents the MR signal present in a given point in space $(x,y,z)$; thus, the affine is a transformation between voxel space and scanner space. Be sure to check [this](https://nipy.org/nibabel/coordinate_systems.html) very good documentation. At the end of section 3 we will discuss a bit more the concept of the image affine.

In [ ]:
print(affine1)

3. The **data** is numpy array where each entry is the value of the reconstructed MR signal. This data however cannot be naively loaded and interpreted without any sort of spatial information.

In [ ]:
print(f"Type: {type(data1)}")
print(f"Shape: {data1.shape}")
print(f"Data:\n {data1}")

Of course, if you overwrite any of these three attributes you can save a new file with the new data. You can do that by simply typing: ```nib.save(nib.Nifti1Header(new_data, new_affine, new_header), new_file_name)```. However, overwiting any of these things without clear knowledge of what you are doing (specially for the header and the affine) could have disastrous consequences. Therefore, it is good practice to use dedicated neuroimaging softwares for certain operations with the data.

## 2. Data visualization: What do I have? How do I see what I have?

As you may have realized, neuroimaging data is complex. Each image has a lot of information that allows any person and/or software to interpret the data correctly. Besides being out of scope, there is not enough time in 1.5h to cover all the technical details around the NIFTI images. Yet, we are now in a position to start taking a proper look into them and, hopefully, provide a comprehensive overview.

Let's start by plotting one scan in the simplest form possible (e.g., the one we have already loaded in memory). Have in mind that each image is a 3D **volume** so we will have to plot slices. 

In [ ]:
fig, ax = plt.subplots(1,3)
ax[0].imshow(data1[70,...], cmap='gray')
ax[0].set_xticks([]), ax[0].set_yticks([]), ax[0].set_title("Sagittal")

ax[1].imshow(data1[:,90,:], cmap='gray')
ax[1].set_xticks([]), ax[1].set_yticks([]), ax[1].set_title("Coronal")

ax[2].imshow(data1[...,125], cmap='gray')
ax[2].set_xticks([]), ax[2].set_yticks([]), ax[2].set_title("Axial")
plt.show()

A faster option is provided by the attributes of the nibabel oject but it does not allow control over which planes to show.

In [ ]:
img1.orthoview()

The first advantage of matplotlib is the control that we have over which plane we wish to show. We can rotate the numpy array, transpose it, select which planes to show, ... something we were not able to do with a DICOM series (i.e., a list of 2D slices). The obvious advantage of using the ```orthoview()``` method is that it is very fast, so it would be useful for quickly inspecting the images. Yet, we can do better with [Nilearn](https://nilearn.github.io/stable/index.html). More specifically, if your intention is to simply visualize a nifti image the [view_image](https://nilearn.github.io/dev/modules/generated/nilearn.plotting.view_img.html) function will output an interactive plot of your scan. The function offers many versatile arguments which may be worth being familiar with.

In [ ]:
plotting.view_img(
    img1, bg_img=False, colorbar=False, cmap='gray', symmetric_cmap=False
)

**Question:** Why is nilearn using the NIFTI object instead of just the numpy array?

This way of plotting images offers the posisbility of overlaying two different data points (e.g., scans from two different subjects). In other type of experiments, where some statistical procedures would have allowed us to identify active parts of the brain in certain conditions, the same function could be used to overlay the brain and its activity. For now, let's simply overlay two images to check the correspondance between them.

**Answer:** The numpy array only holds voxel intensities in voxel indices $(i,j,k)$. To place them in real space (mm) nilearn needs the **affine** (and voxel size / orientation from the header), which are stored only in the NIfTI object. With them it can resample both images to a common grid and overlay them correctly, show L/R and world coordinates - with plain arrays it would just stack two grids that mean different positions.

In [ ]:
# Load data from a second subject
img2 = nib.load("./BIDS-dir/sub-02/anat/sub-02_T1w.nii.gz")
header2, affine2, data2 = img2.header, img2.affine, img2.get_fdata()

plotting.view_img(
    img1, bg_img=img2, opacity=.4,
    colorbar=False, cmap='hot', symmetric_cmap=False
)

As you can see, the two scanners do not overlay between themselves. You may ask the question, "why?" but the answer might not be trivial. A better question is the same one we have asked before: *Why is nilearn using the whole NIFTI object instead of the data to do the plotting?* Let's take a closer look at properties of the objects we have just plotted.

In [ ]:
print("Affine from sub-01")
print(affine1)
print("==============================================")
print("Affine from sub-02")
print(affine2)

What was the image affine codifying? Exactly! The spatial coordinates of each voxel. Let us discover the spatial position of voxel $(60,140,140)$ for each one of the scans. To do so, we simply multiply the affine with the coordinates in voxel space:

In [ ]:
voxel = np.array([60,140,140,1]) # Number 1 is for matrix multiplication
xyz_1 = affine1 @ voxel
xyz_2 = affine2 @ voxel
print("Cartesian coordinates in mm for image 1: ", xyz_1[:-1])
print("Cartesian coordinates in mm for image 2: ", xyz_2[:-1])

It is not surprising that the overlay between the two subjects was not good. Each scan was in its own space (i.e., subject space). Comparing subjects and images is not as straightforward as plotting them together!

All these plots and inspections can be done much easier and (probably) faster with neuroimaging softwares like [FSL](https://fsl.fmrib.ox.ac.uk/fsl/fslwiki), [MRtrix](https://www.mrtrix.org/), [ANTs](https://www.nitrc.org/projects/ants) or [3Dslicer](https://www.slicer.org/). When it comes to plain visualization of nifti objects, it is recommended to have at least one of these three available. In fact, FSL or MRtrix incorporate a lot of useful tools for working with neuroimaging data including but not limited to co-registration and segmentation algorithms. Some of them have overlapping methods while some other have unique software capabilities. Except for 3D slicer, LINUX is the recommended operating system.

## 3. Image registration and tissue segmentation: Unsupervised vs supervised methods

FSL fast and flirt? ANTs? These are softwares that might be cumbersome to install for everybody, but nonetheless, they should be explained. And probably a tour guide through the most important FSL features will come in handy.

Data (pre)processing for neuroimaging includes spatial transformation steps.:
- removal of the non-brain matter (i.e., skull stripping)
- alignment for images that are located in different grids to ensure that the same spatial coordinates denote the same brain region in different patients
- segmenting different types of brain tissue (white matter, gray matter and cerebrospinal fluid) - to be able to restrain certain analyses for specific tissues
- a denoising technique known as removal of bias field homogeneities

Such steps are typically performed with specialized neuroimaging software, that provides resonably accurate results for 3d and 4d images. 
Most popular neuroimaging software for running spatial transformations includes FSL, AFNI and ANTs.
                                                                                                                                                                            
In the following cell, we'll see several examples showcasing FSL's functionality, including:
1. bet (Brain Extraction Tool), a commonly used program to perform brain extraction, which is a type of segmentation used to separate brain (gray matter, white matter and CSF) and non-brain matter (skull, neck)
2. fast (FMRIB's Automated Segmentation Tool), which is a popular implementation for segmenting different brain tissues, usually in three classes: gray matter, white matter and CSF; this procedure is typically performed after non-brain matter was removed from the image (after bet)
3. flirt (FMRIB's Linear Registration Tool), which can run rigid body and affine registration between two images; in this example, we'll perform the registration of an atamocal scan to a standard space. Standard space is a population brain (i.e., and "average" brain) that provides a common reference and coordinate space  to be able to combine data from different participants within the same study, or visualize/analyze results across studies.

For each dataset, depending on the quality of the data, you might need to fine tune some parameters for each step. However, as a benchmark, we will use values that have proven to be robust (based on our judgement) and will comment on other options if they fail.

In [ ]:
%%bash
# Directory and file names
Postprocessing_dir="./BIDS-dir/derivatives/sub-01/anat/"
nifti="./BIDS-dir/sub-01/anat/sub-01_T1w.nii.gz"

# Create derivative directory
mkdir -p $Postprocessing_dir

# 1) Display header information
#fslhd $nifti

# 2) Skull-stripping or brain extraction
# Default bet left neck/skull in (~2.4 L "brain"); -B (bias field + neck cleanup) gives ~1.3 L
masked_brain=$Postprocessing_dir"sub-01_T1w_brain"
bet $nifti $masked_brain.nii.gz -B -f .5 -m
#bet $masked_brain.nii.gz $masked_brain.nii.gz -f .35 -m

# 3) Tissue segmentation (CSF, GM, WM -> *_pve_0/1/2 and *_seg)
#fast -n 4 -I 25 -W 25 -H .25 $masked_brain.nii.gz
fast -t 1 -n 3 $masked_brain.nii.gz

# 4) Registration to template
reference="$FSLDIR/data/standard/MNI152_T1_1mm_brain.nii.gz"
registered=$Postprocessing_dir"sub-01_T1w_brain_space-MNI152"
transformation=$Postprocessing_dir"sub-01_matrix2MNI152.txt"
flirt -in $masked_brain.nii.gz -ref $reference -omat $transformation

flirt -in $masked_brain.nii.gz -ref $reference -out $registered -applyxfm -init $transformation

# Quick checks: brain volume (mm3) and correlation with the template
echo "Brain volume (voxels, mm3): $(fslstats $masked_brain.nii.gz -V)"
echo "Correlation with MNI152: $(fslcc $registered.nii.gz $reference)"

To inspect the outputs of each step you can either do it like in this notebook or by typing ```fsleyes``` into you terminal. Once the viewer has opened, drag the images you want to inspect in the black screen. As you will see, this second option is much more convenient!

All of of the previous methods are unsupervised; i.e., they attemp to minimize a cost function defined on properties of the data. Here are a few examples:

* In the case of ```fast```, mixture gaussian models and hidden markov radiant fields (HMRF) are used. Mixture models allow to define a threshold based on the intensity of the pixels. Added to it, HMRF exploit the fact that two neightbouring models are likely to be of the same type. The ```fast``` command automatically corrects for bias field homogeneities (i.e., the image might look brighter in posterior regions of the brain. Don't worry there are plenty of resources where you can find this and more detailed information: [FAST](https://www.youtube.com/watch?v=xnZoJLSiXTU) (or go to the original references!)
* In the case of ```flirt``` the cost function can be the correlation or the mutual information between the two images (amongst others).
* As in many other examples in machine learning, despite being rather obvious for a human, the ```bet``` algorithm is unintuitively complex. It relies on histogram thresholding and iterative surface estimating.

The only way to become profficient with these steps and algorithms is to fight with them and with some data. Feel free to do so! However, for some data it might be difficult. ANTs might provide more accurate solutions but their tools are substantially heavier in terms of computation. Deep learning might also be a valuable tool. Yet, deep learning usually requires supervised methods with ground-truth labels. Potentially problematic is the "generalizability" of the training and validation dataset. Other possibilities that might improve the final results is the usage of multimodal data.

In [ ]:
from nilearn import plotting, image

d = "./BIDS-dir/derivatives/sub-01/anat"
t1 = "./BIDS-dir/sub-01/anat/sub-01_T1w.nii.gz"
brain = f"{d}/sub-01_T1w_brain.nii.gz"

mask = image.math_img("img > 0", img=brain)

plotting.plot_roi(mask, bg_img=t1, alpha=0.4, cmap="autumn",
                  display_mode="ortho", title="BET – brain mask")
plotting.show()

In [ ]:
mni = os.path.join(os.environ["FSLDIR"], "data/standard/MNI152_T1_1mm_brain.nii.gz")

# fast: 1 = CSF, 2 = GM, 3 = WM
plotting.plot_roi(f"{d}/sub-01_T1w_brain_seg.nii.gz", bg_img=brain,
                  cmap="viridis", alpha=0.5, title="FAST - CSF / GM / WM")

# flirt: registered brain with MNI152 edges on top
display = plotting.plot_anat(f"{d}/sub-01_T1w_brain_space-MNI152.nii.gz",
                             title="FLIRT - sub-01 in MNI152 space")
display.add_edges(mni)
plotting.show()